# HealthCast V3 — Regime-Aware 7-Day Forecasting

V1 and V2 showed that changing tree algorithms alone does not solve the problem. V2 models could pass rolling validation and then fail badly on the final unseen regime. V3 therefore tests one focused hypothesis: **recent-history training + relative-change targets can adapt better to the current regime.**

V3 keeps the same raw dataset, chronological final test, persistence/seasonal-naive baselines, and reliability gate. V1/V2 artifacts are never overwritten.

Experiments: recent windows 60/90/120/180 days × Random Forest / HistGradientBoosting / Ridge × absolute or log-ratio target. A configuration must win at least 3/4 rolling windows and beat the strongest simple baseline on the untouched final test before it can become a production artifact.

In [7]:
# 1 — Setup
from pathlib import Path
import json, warnings, joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
RANDOM_STATE=42; STATE_CODES=["TN","KA","MH","DL","KL"]; HORIZON=7
RECENT_WINDOWS=[60,90,120,180]; N_WINDOWS=4; MIN_TRAIN=60
HERE=Path.cwd(); ROOT=HERE.parent if HERE.name=="notebooks" else HERE
CANDS=[ROOT/"data"/"state_data.csv",HERE/"data"/"state_data.csv",ROOT/"state_data.csv",HERE/"state_data.csv"]
DATA_PATH=next((p for p in CANDS if p.exists()),None)
if DATA_PATH is None: raise FileNotFoundError("Put the uploaded raw file at F:/HealthCast-main/data/state_data.csv. Checked: "+str([str(p) for p in CANDS]))
ARTIFACT_DIR_V3=ROOT/"artifacts_v3"; ARTIFACT_DIR_V3.mkdir(parents=True,exist_ok=True)
print("DATA_PATH:",DATA_PATH); print("V3 ARTIFACTS:",ARTIFACT_DIR_V3)

DATA_PATH: f:\HealthCast-main\state_data.csv
V3 ARTIFACTS: f:\HealthCast-main\artifacts_v3


In [8]:
# 2 — Load raw data and build daily confirmed series
raw=pd.read_csv(DATA_PATH)
print("Shape:",raw.shape); print("Status:"); display(raw["Status"].value_counts(dropna=False))
date_col="Date_YMD" if "Date_YMD" in raw.columns else "Date"
df=raw.copy(); df[date_col]=pd.to_datetime(df[date_col],errors="coerce")
df=df[df["Status"].astype(str).str.strip().str.lower().eq("confirmed")].copy()
for st in STATE_CODES: df[st]=pd.to_numeric(df[st],errors="coerce").fillna(0.0)
df=df[[date_col]+STATE_CODES].dropna(subset=[date_col])
state_daily=df.groupby(date_col)[STATE_CODES].sum().sort_index()
state_daily=state_daily.reindex(pd.date_range(state_daily.index.min(),state_daily.index.max(),freq="D")).fillna(0.0)
state_daily.index.name="Date"
print("Daily table:",state_daily.shape); display(state_daily.head())

Shape: (1524, 42)
Status:


Status
Confirmed    508
Recovered    508
Deceased     508
Name: count, dtype: int64

Daily table: (508, 5)


,TN,KA,MH,DL,KL
Date,,,,,
2020-03-14,1,6,14,7,19
2020-03-15,0,0,18,0,5
2020-03-16,0,1,6,0,3
2020-03-17,0,2,3,1,0
2020-03-18,1,5,3,2,0


## 3. Leakage-safe features and targets
Features use only past information. The relative target is `log1p(future) - log1p(current)`, which is equivalent to predicting the multiplicative change from the current level while remaining numerically stable.

In [9]:
# 3 — Features and direct 7-day targets
FEATURES=["lag_1","lag_2","lag_3","lag_7","lag_14","rolling_avg_7","rolling_avg_14","rolling_std_7","growth_rate","level_ratio_7","level_ratio_14","trend_slope_7","trend_slope_14"]
def slope(v):
    v=np.asarray(v,float)
    if len(v)<2 or not np.all(np.isfinite(v)): return np.nan
    return float(np.polyfit(np.arange(len(v)),v,1)[0])
def make_features(s):
    s=pd.Series(s,dtype=float); f=pd.DataFrame(index=s.index)
    for l in [1,2,3,7,14]: f[f"lag_{l}"]=s.shift(l)
    f["rolling_avg_7"]=s.shift(1).rolling(7).mean(); f["rolling_avg_14"]=s.shift(1).rolling(14).mean(); f["rolling_std_7"]=s.shift(1).rolling(7).std()
    f["growth_rate"]=((s.shift(1)-s.shift(2))/s.shift(2).replace(0,np.nan)).replace([np.inf,-np.inf],np.nan).clip(-5,5)
    f["level_ratio_7"]=s.shift(1)/f["rolling_avg_7"].replace(0,np.nan); f["level_ratio_14"]=s.shift(1)/f["rolling_avg_14"].replace(0,np.nan)
    f["trend_slope_7"]=s.shift(1).rolling(7).apply(slope,raw=True); f["trend_slope_14"]=s.shift(1).rolling(14).apply(slope,raw=True)
    return f
def make_supervised(s):
    s=pd.Series(s,dtype=float); X=make_features(s); cur=s.shift(1).replace(0,np.nan)
    ya=pd.DataFrame({f"h_{h}":s.shift(-h) for h in range(1,HORIZON+1)},index=s.index)
    yr=pd.DataFrame({f"h_{h}":np.log1p(s.shift(-h))-np.log1p(cur) for h in range(1,HORIZON+1)},index=s.index)
    z=X.join(ya,rsuffix="_a").join(yr,rsuffix="_r").replace([np.inf,-np.inf],np.nan).dropna()
    return X.loc[z.index,FEATURES],ya.loc[z.index],yr.loc[z.index]

In [10]:
# 4 — Baselines and metrics
def mae(a,b): return float(np.mean(np.abs(np.asarray(a)-np.asarray(b))))
def rmse(a,b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
def persistence(s,pos): return np.repeat(float(s.iloc[pos]),HORIZON)
def seasonal(s,pos):
    v=s.iloc[:pos+1].to_numpy(float); out=[]
    for h in range(1,HORIZON+1): out.append(v[min(max(len(v)-7+h-1,0),len(v)-1)])
    return np.asarray(out)
def inverse_ratio(r,current): return np.maximum(np.expm1(np.asarray(r)+np.log1p(max(float(current),0))),0)
def model_factory(name):
    if name=="random_forest": return RandomForestRegressor(n_estimators=250,max_depth=12,min_samples_split=5,min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1)
    if name=="hist_gradient_boosting": return MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=250,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=RANDOM_STATE))
    if name=="ridge": return Pipeline([("scale",StandardScaler()),("model",MultiOutputRegressor(Ridge(alpha=10.0)))])
    raise ValueError(name)
def fit_predict(name,target,Xtr,Ya,Yr,Xp):
    m=model_factory(name); y=Ya if target=="absolute" else Yr; m.fit(Xtr,y); return m,m.predict(Xp)

## 5. Diagnose the regime shift
This is retained as evidence, not as a reason to change the test. It records how much the final test level differs from the training level.

In [11]:
# 5 — Train/test regime diagnostic
rows=[]
for st in STATE_CODES:
    s=state_daily[st].astype(float); X,Ya,Yr=make_supervised(s); split=len(X)-max(HORIZON,int(np.ceil(len(X)*.20)))
    tr=s.loc[:X.index[split-1]]; te=s.loc[X.index[split]:]
    rows.append({"state":st,"train_end":str(tr.index[-1].date()),"test_start":str(te.index[0].date()),"train_mean":tr.mean(),"test_mean":te.mean(),"last_train":tr.iloc[-1],"test_median":te.median(),"test_to_train_ratio":te.mean()/max(tr.mean(),1e-9)})
diagnostics_df=pd.DataFrame(rows).set_index("state"); display(diagnostics_df)

,train_end,test_start,train_mean,test_mean,last_train,test_median,test_to_train_ratio
state,,,,,,,
TN,2021-04-20,2021-04-21,2514.585608,14781.657143,10986.0,12772.0,5.878367
KA,2021-04-20,2021-04-21,2974.302730,16298.228571,21794.0,8249.0,5.479680
MH,2021-04-22,2021-04-23,10110.716049,21613.864078,67013.0,10219.0,2.137718
DL,2021-04-20,2021-04-21,2247.000000,5056.285714,28395.0,259.0,2.250238
KL,2021-04-22,2021-04-23,3264.333333,20651.398058,26995.0,17466.0,6.326375


## 6. Rolling-origin V3 experiment
For each validation origin, each candidate trains only on the selected recent window immediately before that origin. The strongest baseline is the lower MAE of persistence and seasonal naive. A candidate must beat that baseline in at least 3 of 4 windows.

In [13]:
# 6 — FAST ROLLING VALIDATION

MODELS = [
    "random_forest",
    "hist_gradient_boosting",
]

TARGETS = [
    "absolute",
    "relative",
]

# Faster model settings for validation.
# Final production fitting can use the selected configuration later.
def build_fast_model(model_name):

    if model_name == "random_forest":
        return RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    if model_name == "hist_gradient_boosting":
        return MultiOutputRegressor(
            HistGradientBoostingRegressor(
                max_iter=120,
                learning_rate=0.05,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=RANDOM_STATE,
            )
        )

    raise ValueError(model_name)


def fast_fit_predict(
    model_name,
    target_mode,
    X_train,
    Y_abs_train,
    Y_rel_train,
    X_pred,
):

    model = build_fast_model(model_name)

    if target_mode == "absolute":

        model.fit(
            X_train,
            Y_abs_train,
        )

        pred = model.predict(X_pred)

    elif target_mode == "relative":

        model.fit(
            X_train,
            Y_rel_train,
        )

        pred = model.predict(X_pred)

    else:
        raise ValueError(target_mode)

    return model, np.asarray(pred, dtype=float)


def origins(n):

    latest = n - HORIZON

    earliest = max(
        MIN_TRAIN,
        latest - (N_WINDOWS - 1) * HORIZON
    )

    if latest <= earliest:
        return []

    return np.linspace(
        earliest,
        latest,
        N_WINDOWS,
        dtype=int
    ).tolist()


def one_fast_eval(
    series,
    X,
    Ya,
    Yr,
    origin,
    window,
    model_name,
    target_mode,
):

    if origin < window:
        return None

    # Recent training window only
    train_start = origin - window
    train_end = origin

    X_train = X.iloc[train_start:train_end]
    A_train = Ya.iloc[train_start:train_end]
    R_train = Yr.iloc[train_start:train_end]

    X_val = X.iloc[
        origin:origin + HORIZON
    ]

    Y_val = Ya.iloc[
        origin:origin + HORIZON
    ]

    if len(X_val) < HORIZON:
        return None

    # Train candidate
    model, pred = fast_fit_predict(
        model_name,
        target_mode,
        X_train,
        A_train,
        R_train,
        X_val,
    )

    y_true = Y_val.to_numpy(float)

    # Convert relative prediction back to cases
    if target_mode == "relative":

        pred = np.vstack([
            inverse_ratio(
                pred[i],
                series.loc[X_val.index[i]]
            )
            for i in range(len(X_val))
        ])

    else:

        pred = np.maximum(
            pred,
            0
        )

    # Persistence baseline
    persistence_pred = np.vstack([
        persistence(
            series,
            series.index.get_loc(d)
        )
        for d in X_val.index
    ])

    # Seasonal naive baseline
    seasonal_pred = np.vstack([
        seasonal(
            series,
            series.index.get_loc(d)
        )
        for d in X_val.index
    ])

    persistence_error = mae(
        y_true,
        persistence_pred
    )

    seasonal_error = mae(
        y_true,
        seasonal_pred
    )

    strongest_baseline = min(
        persistence_error,
        seasonal_error
    )

    model_error = mae(
        y_true,
        pred
    )

    return {
        "window": window,
        "model": model_name,
        "target_mode": target_mode,
        "origin": str(
            X_val.index[0].date()
        ),
        "mae": model_error,
        "rmse": rmse(
            y_true,
            pred
        ),
        "persistence_mae": persistence_error,
        "seasonal_naive_mae": seasonal_error,
        "baseline_mae": strongest_baseline,
        "win": model_error < strongest_baseline,
    }


# ---------------------------------------------------------
# RUN VALIDATION
# ---------------------------------------------------------

rows = []

for st in STATE_CODES:

    print(f"\n========== {st} ==========")

    series = state_daily[st].astype(float)

    X, Ya, Yr = make_supervised(series)

    state_origins = origins(len(X))

    print(
        "Validation origins:",
        len(state_origins)
    )

    total_configs = (
        len(state_origins)
        * len(RECENT_WINDOWS)
        * len(MODELS)
        * len(TARGETS)
    )

    completed = 0

    for origin in state_origins:

        for window in RECENT_WINDOWS:

            for model_name in MODELS:

                for target_mode in TARGETS:

                    result = one_fast_eval(
                        series,
                        X,
                        Ya,
                        Yr,
                        origin,
                        window,
                        model_name,
                        target_mode,
                    )

                    if result is not None:

                        result["state"] = st
                        rows.append(result)

                    completed += 1

                    if completed % 8 == 0:

                        print(
                            f"{completed}/{total_configs}",
                            end="\r"
                        )


rolling_df = pd.DataFrame(rows)

print("\n")
print(
    "Total validation evaluations:",
    len(rolling_df)
)

display(
    rolling_df.head()
)


========== TN ==========
Validation origins: 4
64/64
========== KA ==========
Validation origins: 4
64/64
========== MH ==========
Validation origins: 4
64/64
========== DL ==========
Validation origins: 4
64/64
========== KL ==========
Validation origins: 4
64/64

Total validation evaluations: 320


,window,model,target_mode,origin,mae,rmse,persistence_mae,seasonal_naive_mae,baseline_mae,win,state
0,60,random_forest,absolute,2021-06-30,7105.732655,7763.836478,659.673469,1182.530612,659.673469,False,TN
1,60,random_forest,relative,2021-06-30,59.955796,83.719759,659.673469,1182.530612,659.673469,True,TN
2,60,hist_gradient_boosting,absolute,2021-06-30,5673.161973,6018.871816,659.673469,1182.530612,659.673469,False,TN
3,60,hist_gradient_boosting,relative,2021-06-30,72.781096,96.450453,659.673469,1182.530612,659.673469,True,TN
4,90,random_forest,absolute,2021-06-30,1034.987554,1118.836499,659.673469,1182.530612,659.673469,False,TN


In [14]:
# 7 — Validation summary and selection
summary=(rolling_df.groupby(["state","window","model","target_mode"]).agg(validation_mae=("mae","mean"),validation_rmse=("rmse","mean"),windows_won=("win","sum"),windows_tested=("win","count"),baseline_mae=("baseline_mae","mean")).reset_index())
summary["rolling_pass"]=(summary.windows_won>=3)&(summary.windows_tested==N_WINDOWS)
sel=[]
for st in STATE_CODES:
    q=summary[summary.state==st]; e=q[q.rolling_pass]
    if len(e):
        b=e.loc[e.validation_mae.idxmin()]; sel.append({"state":st,"selected_window":int(b.window),"selected_model":b.model,"selected_target_mode":b.target_mode,"validation_mae":float(b.validation_mae),"windows_won":int(b.windows_won),"windows_tested":int(b.windows_tested),"rolling_pass":True})
    else: sel.append({"state":st,"selected_window":None,"selected_model":None,"selected_target_mode":None,"validation_mae":np.nan,"windows_won":0,"windows_tested":0,"rolling_pass":False})
selection_df=pd.DataFrame(sel).set_index("state"); display(selection_df)

,selected_window,selected_model,selected_target_mode,validation_mae,windows_won,windows_tested,rolling_pass
state,,,,,,,
TN,180.0,random_forest,relative,148.302503,3,4,True
KA,180.0,random_forest,relative,411.289379,3,4,True
MH,NaN,None,None,NaN,0,0,False
DL,NaN,None,None,NaN,0,0,False
KL,NaN,None,None,NaN,0,0,False


## 7. Final untouched test
The selected configuration is retrained on only the selected recent window immediately before the final test. The test is never used for model selection.

In [15]:
# 8 — FINAL TEST ONLY
# IMPORTANT: Rolling validation is already finished.
# Run Cell 10 (#7 — Validation summary and selection) FIRST.
# This cell does NOT run rolling validation again.

if "selection_df" not in globals():
    raise RuntimeError(
        "selection_df is missing. Run Cell 10 (#7 — Validation summary and selection) first. "
        "Do NOT rerun rolling validation."
    )

final = []
predictions = {}

for st in STATE_CODES:
    s = state_daily[st].astype(float)
    X, Ya, Yr = make_supervised(s)

    split = len(X) - max(HORIZON, int(np.ceil(len(X) * 0.20)))

    Xtr, Xte = X.iloc[:split], X.iloc[split:]
    Atr, Are = Ya.iloc[:split], Ya.iloc[split:]
    Rtr = Yr.iloc[:split]
    y = Are.to_numpy(float)

    pv = np.vstack([persistence(s, s.index.get_loc(d)) for d in Xte.index])
    sv = np.vstack([seasonal(s, s.index.get_loc(d)) for d in Xte.index])

    pmae = mae(y, pv)
    smae = mae(y, sv)
    strongest = min(pmae, smae)

    q = selection_df.loc[st]

    if not bool(q["rolling_pass"]):
        final.append({
            "state": st,
            "selected_window": None,
            "selected_model": None,
            "selected_target_mode": None,
            "model_mae": np.nan,
            "model_rmse": np.nan,
            "persistence_mae": pmae,
            "seasonal_naive_mae": smae,
            "strongest_baseline_mae": strongest,
            "beats_strongest_baseline": False,
            "rolling_pass": False,
            "production_gate": False,
        })
        continue

    window = int(q["selected_window"])
    model_name = q["selected_model"]
    target = q["selected_target_mode"]

    m, p = fit_predict(
        model_name,
        target,
        Xtr.iloc[-window:],
        Atr.iloc[-window:],
        Rtr.iloc[-window:],
        Xte,
    )

    if target == "relative":
        p = np.vstack([
            inverse_ratio(p[i], s.loc[Xte.index[i]])
            for i in range(len(Xte))
        ])
    else:
        p = np.maximum(np.asarray(p, dtype=float), 0)

    mmae = mae(y, p)
    mrmse = rmse(y, p)
    beats = mmae < strongest
    gate = bool(beats and q["rolling_pass"])

    final.append({
        "state": st,
        "selected_window": window,
        "selected_model": model_name,
        "selected_target_mode": target,
        "model_mae": mmae,
        "model_rmse": mrmse,
        "persistence_mae": pmae,
        "seasonal_naive_mae": smae,
        "strongest_baseline_mae": strongest,
        "beats_strongest_baseline": beats,
        "rolling_pass": True,
        "production_gate": gate,
    })

    predictions[st] = {
        "model": m,
        "window": window,
        "model_name": model_name,
        "target_mode": target,
        "predictions": p,
        "actual": y,
        "dates": list(Xte.index),
    }

final_test_df = pd.DataFrame(final).set_index("state")

print("V3 FINAL TEST")
display(final_test_df)

passed_states = final_test_df.index[
    final_test_df["production_gate"].fillna(False)
].tolist()

print("Production-passed states:", passed_states)


V3 FINAL TEST


,selected_window,selected_model,selected_target_mode,model_mae,model_rmse,persistence_mae,seasonal_naive_mae,strongest_baseline_mae,beats_strongest_baseline,rolling_pass,production_gate
state,,,,,,,,,,,
TN,180.0,random_forest,relative,6018.457382,10499.572183,2289.447522,4048.709913,2289.447522,False,True,False
KA,180.0,random_forest,relative,7177.809479,15740.792585,3224.900875,5031.026239,3224.900875,False,True,False
MH,NaN,None,None,NaN,NaN,3512.953869,4429.657738,3512.953869,False,False,False
DL,NaN,None,None,NaN,NaN,1225.690962,1922.478134,1225.690962,False,False,False
KL,NaN,None,None,NaN,NaN,4001.514881,3514.711310,3514.711310,False,False,False


Production-passed states: []


In [16]:
# 9 — Horizon and bias diagnostics
hrows=[]; brows=[]
for st,info in predictions.items():
    y=info["actual"]; p=info["predictions"]; s=state_daily[st].astype(float); dates=info["dates"]
    pv=np.vstack([persistence(s,s.index.get_loc(d)) for d in dates]); sv=np.vstack([seasonal(s,s.index.get_loc(d)) for d in dates])
    brows.append({"state":st,"bias":float(np.mean(p-y)),"actual_mean":float(np.mean(y)),"prediction_mean":float(np.mean(p))})
    for h in range(HORIZON): hrows.append({"state":st,"horizon":h+1,"model_mae":mae(y[:,h],p[:,h]),"persistence_mae":mae(y[:,h],pv[:,h]),"seasonal_naive_mae":mae(y[:,h],sv[:,h])})
horizon_df=pd.DataFrame(hrows); bias_df=pd.DataFrame(brows).set_index("state"); display(horizon_df); display(bias_df)

,state,horizon,model_mae,persistence_mae,seasonal_naive_mae
0,TN,1,2402.784719,613.561224,4199.428571
1,TN,2,3920.839302,1201.632653,4151.959184
2,TN,3,5004.126745,1768.438776,4098.795918
3,TN,4,5970.705073,2312.275510,4044.397959
4,TN,5,7113.253162,2847.408163,3995.887755
5,TN,6,8211.274117,3381.561224,3949.244898
6,TN,7,9506.218555,3901.255102,3901.255102
7,KA,1,4183.876284,2011.846939,5399.163265
8,KA,2,4940.058580,2317.367347,5290.408163
9,KA,3,6028.698469,2658.142857,5168.795918


,bias,actual_mean,prediction_mean
state,,,
TN,5972.776178,15223.778426,21196.554603
KA,5705.334055,16290.769679,21996.103734


## 8. Frozen V1/V2 comparison
These are historical results only and are not used to tune V3. V3 still has to beat the simple baseline on the untouched final test.

In [17]:
# 10 — Compare against frozen V1/V2 results
V1={"TN":10497.213733,"KA":11284.369522,"MH":6741.327720,"DL":2494.220057,"KL":12471.667340}
V2={"TN":9445.693522,"KA":7259.864727,"MH":np.nan,"DL":np.nan,"KL":np.nan}
comparison_df=final_test_df.copy(); comparison_df["v1_mae"]=pd.Series(V1); comparison_df["v2_mae"]=pd.Series(V2); comparison_df["v3_vs_v1_pct"]=(comparison_df.v1_mae-comparison_df.model_mae)/comparison_df.v1_mae*100; display(comparison_df)

,selected_window,selected_model,selected_target_mode,model_mae,model_rmse,persistence_mae,seasonal_naive_mae,strongest_baseline_mae,beats_strongest_baseline,rolling_pass,production_gate,v1_mae,v2_mae,v3_vs_v1_pct
state,,,,,,,,,,,,,,
TN,180.0,random_forest,relative,6018.457382,10499.572183,2289.447522,4048.709913,2289.447522,False,True,False,10497.213733,9445.693522,42.666144
KA,180.0,random_forest,relative,7177.809479,15740.792585,3224.900875,5031.026239,3224.900875,False,True,False,11284.369522,7259.864727,36.391577
MH,NaN,None,None,NaN,NaN,3512.953869,4429.657738,3512.953869,False,False,False,6741.327720,NaN,NaN
DL,NaN,None,None,NaN,NaN,1225.690962,1922.478134,1225.690962,False,False,False,2494.220057,NaN,NaN
KL,NaN,None,None,NaN,NaN,4001.514881,3514.711310,3514.711310,False,False,False,12471.667340,NaN,NaN


## 9. Final production decision
A state is production-eligible only when both conditions are true:

1. selected V3 configuration wins at least 3/4 rolling validation windows;
2. selected V3 configuration beats the strongest baseline on the untouched final test.

If zero states pass, V3 remains an experiment and Streamlit must continue to withhold unsupported forecasts.

In [18]:
# 11 — Gate and save V3 artifacts
production_summary=final_test_df[["selected_window","selected_model","selected_target_mode","model_mae","persistence_mae","seasonal_naive_mae","strongest_baseline_mae","beats_strongest_baseline","rolling_pass","production_gate"]].copy(); display(production_summary)
passed_states=production_summary.index[production_summary.production_gate.fillna(False)].tolist(); print("V3 production-passed states:",passed_states)
metadata={"version":"HealthCast_V3_Regime_Aware","horizon":HORIZON,"features":FEATURES,"recent_windows":RECENT_WINDOWS,"models":MODELS,"target_modes":TARGETS,"data_source":str(DATA_PATH),"gate":{"rolling_wins_required":3,"rolling_windows":N_WINDOWS,"final_test_beats_baseline":True},"states":{}}
for st in STATE_CODES:
    r=production_summary.loc[st].to_dict(); r={k:(v.item() if isinstance(v,np.generic) else v) for k,v in r.items()}; r["validation_windows_won"]=int(selection_df.loc[st,"windows_won"]); r["validation_windows_tested"]=int(selection_df.loc[st,"windows_tested"]); metadata["states"][st]=r
    if bool(r["production_gate"]) and st in predictions: joblib.dump(predictions[st]["model"],ARTIFACT_DIR_V3/f"state_model_{st}.pkl")
with open(ARTIFACT_DIR_V3/"model_metadata.json","w",encoding="utf-8") as f: json.dump(metadata,f,indent=2)
state_daily.to_csv(ARTIFACT_DIR_V3/"state_data.csv")
print("Saved:",ARTIFACT_DIR_V3); print("Passed states:",passed_states)

,selected_window,selected_model,selected_target_mode,model_mae,persistence_mae,seasonal_naive_mae,strongest_baseline_mae,beats_strongest_baseline,rolling_pass,production_gate
state,,,,,,,,,,
TN,180.0,random_forest,relative,6018.457382,2289.447522,4048.709913,2289.447522,False,True,False
KA,180.0,random_forest,relative,7177.809479,3224.900875,5031.026239,3224.900875,False,True,False
MH,NaN,None,None,NaN,3512.953869,4429.657738,3512.953869,False,False,False
DL,NaN,None,None,NaN,1225.690962,1922.478134,1225.690962,False,False,False
KL,NaN,None,None,NaN,4001.514881,3514.711310,3514.711310,False,False,False


V3 production-passed states: []
Saved: f:\HealthCast-main\artifacts_v3
Passed states: []


# Final interpretation

V3 is successful only if the evidence supports it. Do not change the test period, weaken the gate, or edit metadata manually. If no state passes, the correct conclusion is that this regime-aware formulation still does not produce a reliable production forecast, and we stop model-chasing rather than forcing a dashboard result.